# Personal AI Tax Adviser

### Loading all the pdfs for rag

In [1]:
!pip install -qU langchain-community pypdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = [
    "/content/drive/MyDrive/pdfs for rag/CBDT_e-Filing_ITR 4_Validation Rules_AY 2026-27.pdf",
    "/content/drive/MyDrive/pdfs for rag/Common ITR Filing FAQs AY 2024-25.pdf",
    "/content/drive/MyDrive/pdfs for rag/Financial Education Booklet - English.pdf",
    "/content/drive/MyDrive/pdfs for rag/ITR-7_FAQ_AY_2024-25.pdf",
    "/content/drive/MyDrive/pdfs for rag/New vs. Old Regime FAQs approved final.pdf",
]

all_documents = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from {path.split('/')[-1]}")
    all_documents.extend(documents)

/tmp/ipykernel_27192/292998271.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 24 pages from CBDT_e-Filing_ITR 4_Validation Rules_AY 2026-27.pdf
Loaded 5 pages from Common ITR Filing FAQs AY 2024-25.pdf
Loaded 73 pages from Financial Education Booklet - English.pdf
Loaded 17 pages from ITR-7_FAQ_AY_2024-25.pdf
Loaded 5 pages from New vs. Old Regime FAQs approved final.pdf


### Split, Embed, and Store

In [3]:
!pip install -qU langchain langchain-core langchain-community langchain-text-splitters \
    langchain-huggingface langchain-chroma pypdf nltk chromadb

In [6]:
#!pip install -U --quiet opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-common opentelemetry-proto chromadb

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(all_documents)
print(f"Total chunks: {len(chunks)}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    collection_name="finance_docs",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db"
)
vector_store.add_documents(documents=chunks)
print("Knowledge base ready!")

Total chunks: 346


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base ready!


###Converting to a Tool

In [5]:
from langchain.tools import tool

@tool
def search_tax_docs(question: str) -> str:
    """Search official income tax documents and SEBI financial education guides for Section 80C/80D deductions, old vs new tax regime comparison, ITR filing guidance, and tax saving investments"""
    results = vector_store.similarity_search(question, k=3)
    if not results:
        return "No relevant information found in the knowledge base."
    context = ""
    for doc in results:
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', 'N/A')
        context += f"Source: {source} (Page {page + 1})\n"
        context += f"Content: {doc.page_content}\n\n"
    return context

###The Full AI Advisor

In [6]:
!pip install -qU langchain-groq

In [7]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

api_key = userdata.get('GROQ_API_KEY')

model = init_chat_model(
    "groq:llama-3.3-70b-versatile",
    api_key=api_key,
)

In [9]:
system_prompt = """You are a Personal Tax Advisor for Indian citizens.

You have access to these tools:
- search_tax_docs: Search official Indian government documents for tax saving
  options (80C, 80D), old vs new tax regime, ITR filing, and investment guidance

Help the user by looking up the relevant data using your tools and giving clear,
specific answers with actual numbers and rates. Always mention the source of
your information. All monetary values should be in Indian Rupees (₹) unless
specified otherwise.
"""

agent = create_agent(
    model=model,
    tools=[search_tax_docs],
    system_prompt=system_prompt,
)

In [10]:
import requests

response = agent.invoke({
    "messages": [{"role": "user", "content": "What's the difference between the old and new tax regime?"}]
})
print("What's the difference between the old and new tax regime?")
print(response["messages"][-1].content)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Can I claim deduction for health insurance premium under Section 80D? What's the limit?"}]
})
print("Can I claim deduction for health insurance premium under Section 80D? What's the limit?")
print(response["messages"][-1].content)


What's the difference between the old and new tax regime?
The key differences between the old and new tax regimes in India are as follows:

1. Tax Rates: The new tax regime has lower tax rates, with tax slabs of 5%, 10%, 15%, 20%, and 25% for different income ranges. In contrast, the old tax regime has tax slabs of 5%, 10%, 15%, 20%, and 30% for different income ranges.

2. Deductions and Exemptions: The new tax regime does not allow for most deductions and exemptions, such as those available under Section 80C, 80D, and HRA. The old tax regime allows for these deductions and exemptions.

3. Tax Liability: Taxpayers can choose between the two regimes and opt for the one that results in lower tax liability.

It is advisable to do a comparative evaluation and analysis under both regimes and then choose as per requirement. Taxpayers can broadly estimate and compare tax liability under the new and the old tax regime using the Income and Tax Calculator on the Income Tax Portal.

Source: FAQs

## Building the UI

In [11]:
!pip install -qU gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00


In [13]:
import gradio as gr

def extract_text(message):
    """Extract text from an LLM message, handling both string and list content formats."""
    return message.text

def tax_advisor(question):
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    last_message = response["messages"][-1]
    return extract_text(last_message)

demo = gr.Interface(
    fn=tax_advisor,
    inputs=gr.Textbox(lines=2, placeholder="Ask a tax question...", label="Question"),
    outputs=gr.Textbox(lines=10, label="Answer"),
    title="Personal Tax AI Advisor",
    description="Ask about Section 80C/80D deductions, old vs new tax regime, ITR filing, and tax saving investments.",
)
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://584bca56c4ed21aa67.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://584bca56c4ed21aa67.gradio.live
